# NB04：手動滴定 vs 自動滴定（Manual vs Automated Titration — Titratable Acidity）

**食品分析實驗 週次 4｜Nielsen's Food Analysis, 6th ed. 對應章節：可滴定酸度（Titratable Acidity）、pH**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 用 KHP 標定 NaOH 濃度：$N = \dfrac{m_{KHP}}{204.22 \times V_L}$。
2. 用 $\%酸 = \dfrac{N \times V \times 當量重}{W \times 1000} \times 100$ 計算不同樣品的可滴定酸度（醋酸/檸檬酸/乳酸/酒石酸）。
3. 讀入 PASCO 匯出的滴定曲線 CSV，用**一階導數最大值**與**二階導數零交叉**找當量點，並理解平滑化(smoothing)的效果。
4. 說明為什麼「不均勻/未內插的體積軸」會讓導數估計出錯，以及該怎麼處理。
5. 用 **Bland–Altman 圖**、成對 t 檢定、F 檢定比較手動滴定與 PASCO 自動滴定，並理解「相關係數 r 高」不代表「兩法一致」。

## 如何使用本筆記本

用法與 NB01–NB03 相同：直接「全部執行」使用內建模擬資料；`load_data(source)` 支援本機路徑、已發布 Google 試算表 CSV、Colab 上傳，或內嵌字串 fallback。本週另外提供 `load_titration_curve(path)`，專門讀取 PASCO 匯出的「體積-pH」滴定曲線 CSV（見第 3 節）。

> ⚠️ **本筆記本內建的資料是模擬教學資料（simulated teaching data）**，滴定曲線由完整電荷平衡式（charge-balance equation）模擬產生（與 W4 課堂投影片使用同一套公式與參數），不是真實儀器輸出。

## 資料格式 (Data Schema)

主要資料沿用共用長格式：`group, student, item, rep, value, unit, temp_C, note, sample`。本週 `item` 包括：`khp_mass_g`/`khp_titration_mL`（NaOH 標定）、`manual_titration_mL`（手動滴定用量）、`pasco_pct_acid`（PASCO 軟體直接算出的 % 酸）。

滴定曲線資料是**另一種格式**（`volume_mL, pH` 兩欄，模擬 PASCO 軟體匯出的 CSV），存在獨立的檔案 `data/nb04_titration_curve.csv`，由專屬的 `load_titration_curve()` 讀取（見第 3 節）。

## 1. 環境設定 (Setup)

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
# ---- Embedded sample data (fallback) ----------------------------------
SAMPLE_CSV = """group,student,item,rep,value,unit,temp_C,note,sample
1,G1_S1,khp_mass_g,1,0.5164,g,25.0,,KHP
1,G1_S1,khp_titration_mL,1,25.08,mL,25.0,,NaOH
2,G2_S1,khp_mass_g,1,0.5215,g,25.0,,KHP
2,G2_S1,khp_titration_mL,1,25.43,mL,25.0,,NaOH
3,G3_S1,khp_mass_g,1,0.4757,g,25.0,,KHP
3,G3_S1,khp_titration_mL,1,23.09,mL,25.0,,NaOH
4,G4_S1,khp_mass_g,1,0.5157,g,25.0,,KHP
4,G4_S1,khp_titration_mL,1,25.09,mL,25.0,,NaOH
5,G5_S1,khp_mass_g,1,0.4777,g,25.0,,KHP
5,G5_S1,khp_titration_mL,1,23.21,mL,25.0,,NaOH
1,G1_S1,manual_titration_mL,1,14.95,mL,25.0,,vinegar
1,G1_S1,pasco_pct_acid,1,4.6,%,25.0,,vinegar
2,G2_S2,manual_titration_mL,2,14.95,mL,25.0,,vinegar
2,G2_S2,pasco_pct_acid,2,4.589,%,25.0,,vinegar
3,G3_S3,manual_titration_mL,3,14.9,mL,25.0,,vinegar
3,G3_S3,pasco_pct_acid,3,4.503,%,25.0,,vinegar
4,G4_S1,manual_titration_mL,1,7.85,mL,25.0,,orange_juice
4,G4_S1,pasco_pct_acid,1,0.995,%,25.0,,orange_juice
5,G5_S2,manual_titration_mL,2,8.0,mL,25.0,,orange_juice
5,G5_S2,pasco_pct_acid,2,1.058,%,25.0,,orange_juice
1,G1_S3,manual_titration_mL,3,7.7,mL,25.0,,orange_juice
1,G1_S3,pasco_pct_acid,3,0.989,%,25.0,,orange_juice
2,G2_S1,manual_titration_mL,1,5.2,mL,25.0,,yogurt
2,G2_S1,pasco_pct_acid,1,0.965,%,25.0,,yogurt
3,G3_S2,manual_titration_mL,2,4.85,mL,25.0,,yogurt
3,G3_S2,pasco_pct_acid,2,0.897,%,25.0,,yogurt
4,G4_S3,manual_titration_mL,3,5.05,mL,25.0,,yogurt
4,G4_S3,pasco_pct_acid,3,0.967,%,25.0,,yogurt
5,G5_S1,manual_titration_mL,1,4.25,mL,25.0,,red_wine
5,G5_S1,pasco_pct_acid,1,0.738,%,25.0,,red_wine
1,G1_S2,manual_titration_mL,2,5.45,mL,25.0,,red_wine
1,G1_S2,pasco_pct_acid,2,0.794,%,25.0,,red_wine
2,G2_S3,manual_titration_mL,3,3.6,mL,25.0,,red_wine
2,G2_S3,pasco_pct_acid,3,0.454,%,25.0,,red_wine
"""

# ---- Embedded PASCO-style titration curve (fallback) -------------------
CURVE_CSV = """Volume (mL),pH
0.006,2.942
0.984,3.544
2.011,3.833
3.014,4.036
3.971,4.181
4.98,4.293
6.002,4.441
6.995,4.523
8.0,4.63
8.987,4.731
10.013,4.826
11.012,4.924
12.001,5.005
13.017,5.105
14.007,5.226
15.987,5.486
16.106,5.513
16.186,5.531
16.313,5.563
16.399,5.609
16.497,5.604
16.59,5.635
16.718,5.699
16.798,5.685
16.894,5.731
16.995,5.724
17.108,5.771
17.205,5.785
17.306,5.829
17.406,5.886
17.532,5.881
17.594,5.948
17.692,5.985
17.788,6.012
17.909,6.057
18.017,6.149
18.098,6.19
18.187,6.271
18.288,6.355
18.41,6.522
18.511,6.642
18.608,6.854
18.69,7.254
18.803,10.092
18.902,10.566
19.003,10.777
19.113,10.923
19.203,11.04
19.31,11.078
19.401,11.153
19.504,11.211
19.609,11.277
19.678,11.29
19.795,11.381
19.893,11.401
19.99,11.411
20.096,11.455
20.222,11.519
20.287,11.547
20.415,11.604
20.475,11.637
20.595,11.615
20.702,11.611
20.809,11.61
20.911,11.678
21.012,11.675
21.095,11.698
21.193,11.711
21.313,11.74
21.397,11.778
21.481,11.772
22.483,11.892
23.486,11.969
24.507,12.032
25.502,12.116
26.51,12.177
27.494,12.257
28.502,12.263
29.509,12.315
"""

In [ ]:
def load_data(source=None):
    """Load NB04 titration/standardization data in the shared long-format
    schema: group, student, item, rep, value, unit, temp_C, note, sample."""
    if source is not None:
        try:
            return pd.read_csv(source)
        except Exception as e:
            print(f"讀取 {source} 失敗，改用內建模擬資料。錯誤訊息：{e}")

    local_path = os.path.join("data", "nb04_sample.csv")
    if os.path.exists(local_path):
        return pd.read_csv(local_path)

    return pd.read_csv(io.StringIO(SAMPLE_CSV))


def load_titration_curve(source=None):
    """Load a PASCO-exported titration curve CSV and return a DataFrame with
    standardized columns ['volume_mL', 'pH'], sorted by volume.

    Tolerant of real-world column-name variants such as 'Volume (mL)',
    'Volume(mL)', 'volume_ml', 'Vol', 'pH ', 'PH', etc.
    """
    if source is not None:
        try:
            raw = pd.read_csv(source)
        except Exception as e:
            print(f"讀取 {source} 失敗，改用內建模擬曲線。錯誤訊息：{e}")
            raw = None
    else:
        raw = None

    if raw is None:
        local_path = os.path.join("data", "nb04_titration_curve.csv")
        if os.path.exists(local_path):
            raw = pd.read_csv(local_path)
        else:
            raw = pd.read_csv(io.StringIO(CURVE_CSV))

    col_map = {}
    for c in raw.columns:
        key = c.strip().lower().replace(" ", "").replace("(ml)", "").replace("_ml", "")
        if "vol" in key:
            col_map[c] = "volume_mL"
        elif key in ("ph", "ph_"):
            col_map[c] = "pH"
    out = raw.rename(columns=col_map)[["volume_mL", "pH"]].dropna()
    out = out.sort_values("volume_mL").reset_index(drop=True)
    return out


df = load_data()
print(f"讀入 {len(df)} 筆資料，共 {df['item'].nunique()} 個量測項目、{df['sample'].nunique()} 種樣品/試劑。")
df.head()

### （選用）在 Colab 上傳班級實際資料

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_data(fname)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_data('本機路徑或URL')")

## 2. NaOH 標定與 % 酸計算

$$N_{NaOH} = \frac{m_{KHP}}{204.22 \times V_L} \qquad \%酸 = \frac{N \times V \times 當量重}{W \times 1000} \times 100$$

### 範例（沿用課程講義數字）

秤 **0.5105 g** 已乾燥的 KHP(MW=204.22)，滴定到終點用去 **24.85 mL** NaOH：

In [ ]:
KHP_MW = 204.22


def khp_normality(khp_g, khp_mL):
    return khp_g / (KHP_MW * khp_mL / 1000.0)


def pct_acid(N, V_mL, eqwt, W):
    return N * V_mL * eqwt / (W * 1000.0) * 100.0


N_demo = khp_normality(0.5105, 24.85)
print(f"N_NaOH = {N_demo:.4f} M  (expect ~0.1006 M)")

# 醋的範例：取醋 2.00 mL，用去 15.1 mL 剛標定好的 NaOH（醋酸當量重 60.05）
pct_demo = pct_acid(N_demo, 15.1, 60.05, 2.00)
print(f"醋酸 % = {pct_demo:.2f}%")

In [ ]:
khp = df[df["item"].isin(["khp_mass_g", "khp_titration_mL"])].pivot_table(
    index="group", columns="item", values="value", aggfunc="first").reset_index()
khp["N_NaOH"] = khp_normality(khp["khp_mass_g"], khp["khp_titration_mL"])
khp

各組 NaOH 都要**自己標定**（NaOH 強吸濕又吸收空氣中 CO₂，濃度不穩定），之後該組所有樣品的 % 酸計算都要用**自己標定出來的 N**，不能直接套用課本或別組的數字。

In [ ]:
EQWT = {"vinegar": 60.05, "orange_juice": 64.04, "yogurt": 90.08, "red_wine": 75.04}
SAMPLE_SIZE = {"vinegar": 2.00, "orange_juice": 5.00, "yogurt": 5.00, "red_wine": 5.00}
SAMPLE_ZH = {"vinegar": "白醋", "orange_juice": "柳橙汁", "yogurt": "優格", "red_wine": "紅酒"}
ACID_ZH = {"vinegar": "醋酸", "orange_juice": "檸檬酸", "yogurt": "乳酸", "red_wine": "酒石酸"}

manual = df[df["item"] == "manual_titration_mL"][["group", "student", "rep", "sample", "value"]].rename(
    columns={"value": "v_manual_mL"})
manual = manual.merge(khp[["group", "N_NaOH"]], on="group")
manual["eqwt"] = manual["sample"].map(EQWT)
manual["W"] = manual["sample"].map(SAMPLE_SIZE)
manual["manual_pct_acid"] = pct_acid(manual["N_NaOH"], manual["v_manual_mL"], manual["eqwt"], manual["W"])
manual["主要酸"] = manual["sample"].map(ACID_ZH)
manual

In [ ]:
manual_summary = manual.groupby("sample")["manual_pct_acid"].agg(n="count", mean_pct="mean", sd_pct="std").reset_index()
manual_summary["rsd_pct"] = manual_summary["sd_pct"] / manual_summary["mean_pct"] * 100
manual_summary

## 3. PASCO 滴定曲線：一階導數找當量點、平滑化的效果

### 理論曲線模型（與 W4 課堂投影片相同的電荷平衡式）

本筆記本用來產生模擬曲線、並在下面拿來與實驗數據比較的「理論當量點」，是用完整的電荷平衡式（charge balance）對 $\log[H^+]$ 求根算出來的，跟 W4 投影片用的是同一組參數（0.075 M 醋酸 25 mL，以 0.1 M NaOH 滴定，$K_a=1.75\times10^{-5}$），**不是**分段近似。

In [ ]:
from scipy.optimize import brentq

Ca, Va, Cb, Ka, Kw = 0.075, 25.0, 0.1, 1.75e-5, 1e-14
pKa = -np.log10(Ka)
Veq = Ca * Va / Cb  # theoretical equivalence volume (mL)


def pH_exact(Vb):
    """Charge balance: Cb0 + [H+] = Ka*Ca0/(Ka+[H+]) + Kw/[H+], solved for
    log[H+] with brentq. Smooth across the whole titration."""
    Vtot = (Va + Vb) / 1000.0
    Ca0 = Ca * Va / 1000.0 / Vtot
    Cb0 = Cb * Vb / 1000.0 / Vtot

    def f(logH):
        H = 10 ** logH
        A = Ka * Ca0 / (Ka + H)
        OH = Kw / H
        return (Cb0 + H) - (A + OH)

    return -brentq(f, -14, 1)


print(f"pKa = {pKa:.3f}, 理論當量點 Veq = {Veq:.2f} mL")

匯出的原始曲線資料**體積間距不均勻**（接近當量點附近，真正的自動滴定儀會自動放慢加液速度、變成更細的間距；本模擬刻意在 14–16 mL 之間留了一段特別粗的間距，示範「間距不均勻」時直接假設固定間距會出錯）。

In [ ]:
curve = load_titration_curve()
print(f"讀入 {len(curve)} 個資料點，體積範圍 {curve['volume_mL'].min():.2f}–{curve['volume_mL'].max():.2f} mL")
curve.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(curve["volume_mL"], curve["pH"], "o-", markersize=3, color="tab:blue")
ax.set_xlabel("Volume of NaOH added (mL)")
ax.set_ylabel("pH")
ax.set_title("Raw exported titration curve (uneven volume spacing)")
plt.tight_layout()
plt.show()

### 錯誤示範：假設固定間距

如果直接用 `np.diff(pH)` 除以一個「假設的」固定體積間距（例如整條曲線的平均間距），每一段的斜率都會被放大或縮小：間距比平均小的地方斜率被**低估**，間距比平均大的地方被**高估**。這條曲線在 14–16 mL 只有一個相隔約 2 mL 的點，錯誤作法會在那裡算出一個**假的凸起**；當量點附近間距只有約 0.1 mL，真正的峰則被壓矮。正確做法一定要用**每一段實際的 ΔV**。

> 這份資料的峰剛好落在同一段，所以兩種作法找到的峰值**位置**一樣，但**峰高**和其他區段的形狀都錯了。如果粗間距剛好落在當量點附近，或假凸起比真峰還高，連位置都會找錯。


In [ ]:
V = curve["volume_mL"].to_numpy()
pH = curve["pH"].to_numpy()

# WRONG: assume a single average step for the whole curve
avg_step_wrong = (V[-1] - V[0]) / (len(V) - 1)
d_wrong = np.diff(pH) / avg_step_wrong
V_mid_wrong = (V[:-1] + V[1:]) / 2
wrong_peak_V = V_mid_wrong[np.argmax(d_wrong)]

# RIGHT: use the actual (uneven) delta-V at each interval
d_right = np.diff(pH) / np.diff(V)
V_mid_right = (V[:-1] + V[1:]) / 2
right_peak_V = V_mid_right[np.argmax(d_right)]

print(f"理論當量點 Veq = {Veq:.2f} mL")
print(f"錯誤作法（假設固定間距）估計的峰值位置：V = {wrong_peak_V:.2f} mL, 峰值 = {d_wrong.max():.2f}")
print(f"正確作法（用實際 ΔV）估計的峰值位置：    V = {right_peak_V:.2f} mL, 峰值 = {d_right.max():.2f}")

# where the coarse 14–16 mL step sits, compare the two slope estimates
coarse = np.argmax(np.diff(V))
print(f"\n最粗的一段：{V[coarse]:.2f}–{V[coarse + 1]:.2f} mL（ΔV = {np.diff(V)[coarse]:.2f} mL）")
print(f"  正確斜率 = {d_right[coarse]:.3f} pH/mL，錯誤作法 = {d_wrong[coarse]:.3f} pH/mL（假凸起）")


### 內插到均勻網格 + 平滑化

把原始資料**內插(interpolate)**到一個細的均勻網格上，再取導數，可以避免「間距不均勻」造成的假象；搭配簡單的移動平均**平滑化(smoothing)**，可以降低雜訊對峰值位置的影響（但平滑太多也會讓峰變寬、變矮，需要取捨）。

In [ ]:
V_grid = np.arange(V.min(), V.max(), 0.05)
pH_interp = np.interp(V_grid, V, pH)

dpH_interp = np.gradient(pH_interp, V_grid)
peak_V_interp = V_grid[np.argmax(dpH_interp)]

# simple moving-average smoothing before differentiating
window = 9
kernel = np.ones(window) / window
pH_smooth = np.convolve(pH_interp, kernel, mode="same")
dpH_smooth = np.gradient(pH_smooth, V_grid)
peak_V_smooth = V_grid[np.argmax(dpH_smooth)]

print(f"內插後（未平滑）一階導數峰值：V = {peak_V_interp:.2f} mL")
print(f"內插 + 平滑後一階導數峰值：    V = {peak_V_smooth:.2f} mL")
print(f"理論當量點：Veq = {Veq:.2f} mL")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(V_grid, dpH_interp, color="tab:orange", label="unsmoothed")
axes[0].plot(V_grid, dpH_smooth, color="tab:green", label="smoothed")
axes[0].axvline(Veq, color="grey", linestyle="--", label="true Veq")
axes[0].set_xlabel("Volume (mL)"); axes[0].set_ylabel("dpH/dV")
axes[0].set_title("First derivative: smoothing effect")
axes[0].legend(fontsize=8)

d2pH = np.gradient(dpH_smooth, V_grid)
axes[1].plot(V_grid, d2pH, color="tab:red")
axes[1].axhline(0, color="grey", linewidth=1)
axes[1].axvline(Veq, color="grey", linestyle="--")
axes[1].set_xlabel("Volume (mL)"); axes[1].set_ylabel("d2pH/dV2")
axes[1].set_title("Second derivative (zero-crossing = equivalence point)")
plt.tight_layout()
plt.show()

### 二階導數零交叉點：要找「真峰旁邊」那一個

平滑後的二階導數在雜訊區會反覆穿過 0，所以不能取搜尋範圍內的**第一個**零交叉點（那通常只是雜訊）。正確的做法是：先用一階導數找到峰，再找峰旁邊二階導數**由正轉負**的那一點；它應該和一階導數峰值幾乎一樣。


In [ ]:
# find the second-derivative zero crossing that belongs to the first-derivative peak.
# Noise makes d2 cross zero many times, so take the + -> - crossing closest to the peak
i_peak = int(np.argmax(dpH_smooth))
down = np.where((d2pH[:-1] > 0) & (d2pH[1:] <= 0))[0]  # + -> - crossings
n_noise = int(((V_grid[:-1] > Veq - 3) & (V_grid[:-1] < Veq + 3) &
               (np.diff(np.sign(d2pH)) != 0)).sum())
if len(down) > 0:
    i = int(down[np.argmin(np.abs(down - i_peak))])
    # linear interpolation between the two straddling points
    V0, V1 = V_grid[i], V_grid[i + 1]
    y0, y1 = d2pH[i], d2pH[i + 1]
    V_zero = V0 - y0 * (V1 - V0) / (y1 - y0)
    print(f"Veq±3 mL 內，二階導數穿過 0 共 {n_noise} 次（大多是雜訊）")
    print(f"一階導數峰值：V = {V_grid[i_peak]:.3f} mL")
    print(f"峰旁邊的二階導數零交叉點：V = {V_zero:.3f} mL  (理論 Veq = {Veq:.2f} mL)")
else:
    print("沒有找到由正轉負的零交叉點，請檢查平滑視窗或資料。")


### 半當量點 pH ≈ pKa

In [ ]:
V_half = Veq / 2.0
pH_half = np.interp(V_half, V_grid, pH_interp)
print(f"半當量點 V = {V_half:.2f} mL -> 內插 pH = {pH_half:.2f}  (理論 pKa = {pKa:.2f})")

## 4. 手動 vs PASCO：Bland–Altman、配對 t 檢定、F 檢定

同一批樣品，手動滴定與 PASCO 自動滴定各測 3 重複，比較兩法是否一致。**注意**：相關係數 r 很高不代表兩法沒有偏差——這正是 Bland–Altman 圖存在的理由。

In [ ]:
pasco = df[df["item"] == "pasco_pct_acid"][["group", "student", "rep", "sample", "value"]].rename(
    columns={"value": "pasco_pct_acid"})
paired = manual.merge(pasco, on=["group", "student", "rep", "sample"])
paired["diff_pct"] = paired["pasco_pct_acid"] - paired["manual_pct_acid"]
paired["mean_pct"] = (paired["pasco_pct_acid"] + paired["manual_pct_acid"]) / 2
paired[["sample", "group", "manual_pct_acid", "pasco_pct_acid", "diff_pct"]]

In [ ]:
bias = paired["diff_pct"].mean()
sd_diff = paired["diff_pct"].std(ddof=1)
loa_hi = bias + 1.96 * sd_diff
loa_lo = bias - 1.96 * sd_diff
n_pairs = len(paired)

t_stat, p_value = stats.ttest_rel(paired["pasco_pct_acid"], paired["manual_pct_acid"])
r_value = np.corrcoef(paired["manual_pct_acid"], paired["pasco_pct_acid"])[0, 1]

print(f"bias (PASCO - manual) = {bias:+.4f}%")
print(f"95% LoA = [{loa_lo:.4f}, {loa_hi:.4f}]  (sd={sd_diff:.4f}, n={n_pairs})")
print(f"paired t-test: t = {t_stat:.3f}, p = {p_value:.4f}")
print(f"correlation r = {r_value:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
colors = {"vinegar": "tab:blue", "orange_juice": "tab:orange", "yogurt": "tab:green", "red_wine": "tab:purple"}
for sample, c in colors.items():
    sub = paired[paired["sample"] == sample]
    ax.scatter(sub["mean_pct"], sub["diff_pct"], s=60, color=c, label=sample)

ax.axhline(bias, color="black", linewidth=1.5, label=f"bias={bias:+.3f}%")
ax.axhline(loa_hi, color="tab:red", linestyle="--", linewidth=1.2, label=f"+1.96SD={loa_hi:.3f}")
ax.axhline(loa_lo, color="tab:red", linestyle="--", linewidth=1.2, label=f"-1.96SD={loa_lo:.3f}")
ax.set_xlabel("Mean of manual and PASCO (%acid)")
ax.set_ylabel("Difference: PASCO - manual (%acid)")
ax.set_title("Bland-Altman: manual vs PASCO titratable acidity")
ax.legend(fontsize=8, loc="best")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for sample, c in colors.items():
    sub = paired[paired["sample"] == sample]
    ax.scatter(sub["manual_pct_acid"], sub["pasco_pct_acid"], s=60, color=c, label=sample)
lims = [paired[["manual_pct_acid", "pasco_pct_acid"]].min().min(),
        paired[["manual_pct_acid", "pasco_pct_acid"]].max().max()]
ax.plot(lims, lims, "k--", linewidth=1, label="y = x")
ax.set_xlabel("Manual %acid")
ax.set_ylabel("PASCO %acid")
ax.set_title(f"High correlation (r={r_value:.3f}) despite a systematic bias")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### F 檢定：比較兩方法「自身重複」的精密度

F 檢定看的是**各方法自己在同一樣品重複測定的變異數**是否相等，跟上面「配對差異」是不同的問題（一個看準確度/偏差，一個看精密度）。

In [ ]:
def pooled_variance(frame, value_col):
    num = den = 0.0
    for _, g in frame.groupby("sample"):
        v = g[value_col].to_numpy()
        num += ((v - v.mean()) ** 2).sum()
        den += len(v) - 1
    return num / den, den


var_manual, df_manual = pooled_variance(paired, "manual_pct_acid")
var_pasco, df_pasco = pooled_variance(paired, "pasco_pct_acid")
F_stat = max(var_manual, var_pasco) / min(var_manual, var_pasco)
print(f"pooled variance: manual = {var_manual:.5f} (df={df_manual}), PASCO = {var_pasco:.5f} (df={df_pasco})")
print(f"F = {F_stat:.3f}")
# two-sided F test: is either method's repeatability different?
if var_manual >= var_pasco:
    df_num, df_den = df_manual, df_pasco
else:
    df_num, df_den = df_pasco, df_manual
p_F = min(1.0, 2 * stats.f.sf(F_stat, df_num, df_den))
F_crit = stats.f.ppf(0.975, df_num, df_den)
print(f"F 臨界值（雙尾 α=0.05，df={df_num:.0f}, {df_den:.0f}）= {F_crit:.3f}，p = {p_F:.3f}")
if F_stat > F_crit:
    print("F > 臨界值：兩方法的精密度有顯著差異。")
else:
    print("F ≤ 臨界值：沒有證據顯示兩方法的精密度不同。")


## 5. 練習題 (Exercises)

### 問題 1：標定與 % 酸計算

某組秤 **0.4890 g** KHP，滴定用去 **23.60 mL** NaOH，求 N<sub>NaOH</sub>。接著用這個 N 滴定 **5.00 mL** 柳橙汁，用去 **7.85 mL** NaOH，求柳橙汁的可滴定酸度 %（以檸檬酸表示，當量重 64.04）。

In [ ]:
# TODO: 請在這裡作答

### 問題 2：為什麼不能假設固定間距

參考第 3 節「錯誤示範」的結果：(1) 錯誤作法和正確作法的峰高差多少？(2) 在 14–16 mL 那段粗間距，錯誤作法算出的斜率是正確值的幾倍？如果這個假凸起剛好比真正的峰還高，會發生什麼事？(3) 提出至少一種修正方法。


In [ ]:
# TODO: 請在這裡作答

### 問題 3：半當量點與 pKa

用第 3 節算出的半當量點 pH，判斷這條模擬曲線最可能對應哪一種酸（提示：查課本或講義中各種酸的 pKa 值），並說明推論依據。

In [ ]:
# TODO: 請在這裡作答

### 問題 4：解讀 Bland–Altman

用第 4 節的結果，回答：(1) 手動與 PASCO 之間是否有系統性偏差(bias)？(2) 哪一個樣品的差異點分布最開？可能的原因是什麼？(3) 如果只看相關係數 r，會不會漏掉重要的訊息？

In [ ]:
# TODO: 請在這裡作答

### 問題 5：t 檢定 vs F 檢定

配對 t 檢定與 F 檢定分別回答什麼問題？如果 t 檢定顯著、但 F 檢定不顯著，該怎麼解讀兩方法的差異？

In [ ]:
# TODO: 請在這裡作答

## 6. 匯出結果 (Export)

In [ ]:
output_path = "nb04_titration_results.csv"
paired.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")